# Step 5 — Tracing & Evaluation

"It answered something" is not the same as "it works". For every test question in `eval/questions.json` we record what the agent did and score it:

| Metric | Question it answers |
|---|---|
| **tool accuracy** | Did it call the tool we expected? |
| **source hit** | Did its searches retrieve the document that holds the answer? |
| **key facts** | Does the answer contain the key fact (e.g. "24")? |
| **faithfulness** | LLM-as-judge: is every claim supported by what the tools returned? |

### Setup
Run this cell first. It lets the notebook import the lab's own code (`config.py`, `rag/`, `agent/`, `llm.py`) from the folder above `notebooks/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Project folder:", ROOT)

In [ ]:
from rag.ingest import build_index

build_index(verbose=False)

In [ ]:
from llm import check_llm

ok, message = check_llm()
print("LLM ready ✅" if ok else f"LLM NOT ready ❌  {message}")

## 1. The test questions

In [ ]:
import json
import config

questions = json.loads((config.PROJECT_ROOT / "eval" / "questions.json").read_text())
for i, q in enumerate(questions, start=1):
    print(f"{i:>2}. {q['question']}   → expect {q['expected_tool']}")

## 2. Pick the agent and how many questions to run
The full set takes a few minutes on a laptop. Start with 3, then run all 10.

In [ ]:
AGENT = "scratch"   # "scratch" (Step 3) or "graph" (Step 4)
N_QUESTIONS = 3     # set to len(questions) for the full run

## 3. Run and score
`evaluate()` and `print_summary()` come from `steps/step5_evaluate.py`, the same code the script uses.

In [ ]:
from steps.step5_evaluate import evaluate, print_summary

if AGENT == "scratch":
    from agent.react_agent import run_react_agent as run_agent
else:
    from agent.graph_agent import run_graph_agent as run_agent

rows = []
for i, item in enumerate(questions[:N_QUESTIONS], start=1):
    print(f"[{i}/{N_QUESTIONS}] {item['question']}")
    rows.append(evaluate(item, run_agent))

print_summary(rows)

## 4. Look at one result in detail

In [ ]:
rows[0]

## Discuss
- The judge is the **same small model** that answered. Why is that a weakness? (That's why we also check key facts.)
- Which metric would you trust most, and why?